In [19]:
%matplotlib widget

import matplotlib.pyplot as plt
import numpy as np
from android_bot.android_capture import ScreenCapture

from android_bot import image_utils
from android_bot import android_actions
import pandas as pd

from android_bot.android_actions import AndroidBJTabletActor

from android_bot import get_cards_tablet
import time

from android_bot.tablet_treewalker_player import TabletPlayer

import logging
from datetime import datetime
import os
from blackjack_cpp import ProbabilisticRankShoe, load_combo_data

import pickle
from blackjack.ev_estimator import EVEstimator
from sklearn.linear_model import LinearRegression
from android_bot.tablet_treewalker_player import TWDecisionMaker
from blackjack_cpp import RandomSampler



combo_path = "../combinations//"
abs_combo_path = os.path.abspath(combo_path)
load_combo_data(combo_path)

In [20]:
# Create logs directory if it doesn't exist
os.makedirs('../logs', exist_ok=True)

# Create log filename with timestamp
log_filename = f"../logs/tw_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()  # This keeps console output as well
    ]
)

print(f"Logging to {log_filename}")

Logging to ../logs/tw_log_20251227_195308.txt


In [21]:
logger = logging.getLogger(__name__)


In [22]:
def bet_size_ramp(tc_int):
    if tc_int >= 6:
        return 20
    elif tc_int >= 5:
        return 18
    elif tc_int >= 4:
        return 14
    elif tc_int >= 3:
        return 10
    elif tc_int >= 2:
        return 6
    elif tc_int >= 1:
        return 3
    else:
        return 1

In [23]:
ad_images = []

def close_ads(actor, src_taker, logger):
    logger.info("Closing ads")

    ignore_region_xy = [[1700,1780], [1350, 1430]]
    def not_in_ignore_region(xy_position):
        return not (
            ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
            and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
        )

    game_img = src_taker.get_screen()
    ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
    ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

    while len(ads_crosses) > 0:
        xy_position = ads_crosses[0]
        actor.click(*xy_position)
        ad_images.append(game_img)
        time.sleep(1)
        game_img = src_taker.get_screen()
        ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
        ads_crosses = list(filter(not_in_ignore_region, ads_crosses))


def restart_table(actor, src_taker, logger):
    logger.info("Restarting table")
    
    # leave table
    actor.leave_table()

    game_img = src_taker.get_screen()
    leave_table_btns = get_cards_tablet.find_leave_table_button(game_img)
    while len(leave_table_btns) == 0:
        time.sleep(1)
        game_img = src_taker.get_screen()
        leave_table_btns = get_cards_tablet.find_leave_table_button(game_img)
    exit_btn = leave_table_btns[0]
    actor.click(*exit_btn)
    
    while True:
        time.sleep(1)
        close_ads(actor, src_taker, logger)
        
        game_img = src_taker.get_screen()
        if get_cards_tablet.can_create_private_table(game_img):
            logger.info("Creating private table")
            actor.create_private_table()
            
            time.sleep(2)
            game_img = src_taker.get_screen()
            if get_cards_tablet.can_create_private_table(game_img):
                # if still can create then try again
                continue
            else:
                break
        elif get_cards_tablet.is_table_empty(game_img):
            return

In [24]:
play_log_filename = f"../logs/tw_play_{datetime.now().strftime("%Y_%m_%d_%H_%M_%S")}.csv"

# columns
# dealer_hand, player_hand_0, player_hand_1, total_bet, net_value, datetime up to second

def create_header_if_not_exists():
    if not os.path.exists(play_log_filename):
        with open(play_log_filename, 'w') as f:
            line = ",".join([
                "shoe_hand_idx",
                "dealer_hand",
                "player_hand_0",
                "player_hand_1",
                "total_bet",
                "net_value",
                "ev_percent",
                "ev_bet",
                "n2", "n3", "n4", "n5", "n6", "n7", "n8", "n9", "n10", "n11",
                "datetime",
            ])
            f.write(line + "\n")

create_header_if_not_exists()

def hand_to_str_line(hand):
    return " ".join(str(rank) for rank in hand.cards)

def log_play_stats(shoe_hand_idx, round, ev_percent, rank_count):
    dealer_hand_str = hand_to_str_line(round.get_dealer_hand())
    player_hand_0_str = hand_to_str_line(round.player_hands[0])
    if len(round.player_hands) > 1:
        player_hand_1_str = hand_to_str_line(round.player_hands[1])
    else:
        player_hand_1_str = ""
    total_bet = round.get_player_bet()
    round_value = round.get_player_value()

    now = datetime.now()
    dt_string = now.strftime("%Y-%m-%d %H:%M:%S")

    with open(play_log_filename, 'a') as f:
        line = ",".join([
            str(shoe_hand_idx),
            dealer_hand_str,
            player_hand_0_str,
            player_hand_1_str,
            str(total_bet),
            str(round_value),
            str(ev_percent),
            str(ev_percent / 100 * total_bet),
            *[str(rank_count.get(i, 0)) for i in range(2, 12)],
            dt_string
        ])
        f.write(line)
        f.write("\n")


In [25]:

src_taker = None
del src_taker

all_cards = []
net_value = []

src_taker = ScreenCapture()  # save_folder="/media/maxim/T7/frames/")
game_img = src_taker.get_screen()
initial_penetration = 0 # get_cards_tablet.get_shoe_penetration(game_img)
logger.info(f"initial_penetration {initial_penetration}")
shoe = ProbabilisticRankShoe(6)
actor = AndroidBJTabletActor([250, 500, 1000, 2500, 5000])


decision_maker = TWDecisionMaker(shoe)

# shoe_rank_count = {
#     a: b for a, b in zip(
#         [2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
#         [22, 16, 15, 17, 11, 18, 20, 21, 72, 20]
#     )
# }
# for r, c in shoe_rank_count.items():
#     shoe.set_number_of_rank_cards(r, c)


2025-12-27 19:53:08,597 - __main__ - INFO - initial_penetration 0


In [26]:
model_path = "../sklearn_models/"

ev_model = pickle.load(open(model_path + "ev_estimator_cubic.pkl", "rb"))
count_model = pickle.load(open(model_path + "synthetic_count_from_ev.pkl", "rb"))

In [27]:
# restart_table(actor, src_taker)

In [28]:
shoe_hand_idx = 0

In [29]:
p = TabletPlayer(actor, src_taker, decision_maker)

In [30]:
baseline_ev_estimate = ev_model.predict(ProbabilisticRankShoe(6).get_rank_count())

In [31]:
while True:
    shoe = decision_maker.get_shoe()
    ev_estimate = ev_model.predict(shoe.get_rank_count()).item()

    logger.info(f"start, ev={ev_estimate}")
    logger.info(f"shoe: {str(shoe)}")
    
    synth_count = count_model.predict(np.atleast_2d(ev_estimate)).item()
    int_tc = int(synth_count)

    bet = bet_size_ramp(int_tc) * 250

    p.play_round(bet)

    for hand in p.round.player_hands:
        all_cards.extend(hand.cards)

    round_value = p.round.get_player_value()
    all_cards.extend(p.round.dealer_hand.cards)
    net_value.append(round_value)
    
    shoe = decision_maker.get_shoe()
    log_play_stats(shoe_hand_idx, p.round, ev_estimate, shoe.get_rank_count())
    
    shoe_hand_idx += 1
    
    next_ev_estimate = ev_model.predict(shoe.get_rank_count()).item()

    if next_ev_estimate < baseline_ev_estimate:
        logger.info(f"ev too low ev_estimate={next_ev_estimate}")
        restart_table(actor, p, logger)
        decision_maker = TWDecisionMaker(ProbabilisticRankShoe(6))
        shoe_hand_idx = 0
    
    elif get_cards_tablet.is_pre_shuffle(p.get_screen()):
        logger.info("shoe exhausted")
        restart_table(actor, p, logger)
        decision_maker = TWDecisionMaker(ProbabilisticRankShoe(6))
        shoe_hand_idx = 0
    
    elif round_value < 0:
        p.rebuy()

    time.sleep(1)

2025-12-27 19:53:10,593 - __main__ - INFO - start, ev=-0.27934664237977636
2025-12-27 19:53:10,594 - __main__ - INFO - shoe: ProbabilisticRankShoe: 
 2,  3,  4,  5,  6,  7,  8,  9, 10, 11
24, 24, 24, 24, 24, 24, 24, 24, 96, 24 (total: 312)

2025-12-27 19:53:10,618 - android_bot.android_actions - INFO - click at 1520.0, 1300.0
2025-12-27 19:53:12,133 - android_bot.android_actions - INFO - click at 1125.0, 800.0
2025-12-27 19:53:13,566 - android_bot.android_actions - INFO - click at 1300.0, 765.0
2025-12-27 19:53:14,490 - android_bot.android_actions - INFO - deal
2025-12-27 19:53:14,491 - android_bot.android_actions - INFO - click at 2140.0, 1300.0
2025-12-27 19:53:14,978 - android_bot.tablet_treewalker_player - INFO - wait_for_initial_cards
2025-12-27 19:53:20,799 - android_bot.tablet_treewalker_player - INFO - initial hand
2025-12-27 19:53:20,802 - android_bot.tablet_treewalker_player - INFO - Dealer A(1/11)
Player 10,5(15)[$250]


RuntimeError: Best action not available after processing all events

In [39]:
print(decision_maker.tree_walker.get_state_info())

TreeWalker State:
  Reference round:
Dealer AX
Player T5(15)[$250]
  Expected: player_action
  Shoe: ProbabilisticRankShoe: 
 2,  3,  4,  5,  6,  7,  8,  9, 10, 11
24, 24, 24, 23, 24, 24, 24, 24, 95, 23 (total: 309)

  Node value: -162.354729 [-162.354729, -158.270780]
  Children built: yes
  Children events and values:
    TAKE_INSURANCE: -172.063466 [-172.063466, -167.979518]
    REFUSE_INSURANCE: -162.354729 [-162.354729, -158.270780]



In [32]:
decision_maker._worker_thread.is_alive()

True

In [33]:
decision_maker._worker_exception

In [34]:
from android_bot.tablet_treewalker_player import drain_and_mark_done

In [35]:
print(decision_maker._event_queue.qsize())


0


In [36]:
print(str(decision_maker.round))

Dealer A(1/11)
Player 10,5(15)[$250]


In [40]:
decision_maker.round.get_stage()

<BJStage.PLAYER_OFFERED_INSURANCE: 2>

In [ ]:
# TreeWalker State:
#   Reference round:
# Dealer AX
# Player 52(7)[$250]
#   Expected: player_action
#   Shoe: ProbabilisticRankShoe: 
#  2,  3,  4,  5,  6,  7,  8,  9, 10, 11
# 18, 17, 18, 16, 18, 12, 14, 20, 74, 15 (total: 222)

#   Node value: -133.508716 [-133.508716, -133.508716]
#   Children built: yes
#   Children events and values:
#     TAKE_INSURANCE: -133.508716 [-133.508716, -133.508716]
#     REFUSE_INSURANCE: -133.508716 [-133.508716, -133.508716]

In [ ]:
close_ads(actor, src_taker)

In [ ]:
ignore_region_xy = [[1700,1780], [1350, 1430]]
def not_in_ignore_region(xy_position):
    return not (
        ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
        and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
    )

game_img = src_taker.get_screen()
ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

while len(ads_crosses) > 0:
    for xy_position in ads_crosses:
        actor.click(*xy_position)
    time.sleep(2)
    game_img = src_taker.get_screen()
    ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
    ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

In [ ]:
ads_crosses

In [ ]:
ignore_region_xy = [[1700,1780], [1350, 1430]]
def not_in_ignore_region(xy_position):
    return not (
        ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
        and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
    )

game_img = src_taker.get_screen()
ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

In [ ]:
ads_crosses

In [ ]:
from blackjack.actions import PlayerAction
from blackjack.blackjack_round import BJRound

round = BJRound(Player.rules)
round.start_round(bet_unit=100)
round.take_card(11)
round.take_card(11)
round.take_card(3)


round.take_action(PlayerAction.SPLIT)

print(round)
round.take_card(2)
round.take_card(10)

print(round)
round.take_action(PlayerAction.STAND)
print(round)

print(round.active_hand_idx)

In [ ]:
p.start_screen_capture()

In [ ]:
from android_bot.image_assets import ImageAssets


game_img = p.get_screen()
empty_table = ImageAssets.empty_table
empty_table_area = (slice(550, 705), slice(1000, 1375))
area_img = game_img[empty_table_area]
match = image_utils.get_best_match(
    area_img, empty_table
)

In [ ]:
match, empty_table.shape

In [ ]:
f, ax = plt.subplots(1, 2)
ax[0].imshow(area_img)
ax[1].imshow(empty_table)
plt.show()

In [ ]:
image_utils.save_rgb_png(area_img, "../ui_elements/empty_table_2.png")

In [ ]:
ignore_region_xy = [[1700,1780], [1350, 1430]]
def not_in_ignore_region(xy_position):
    return not (
        ignore_region_xy[0][0] < xy_position[0] < ignore_region_xy[0][1]
        and ignore_region_xy[1][0] < xy_position[1] < ignore_region_xy[1][1]
    )

game_img = ad_images[0]  # image_utils.load_rdb("../frames_tablet/leave_table_grey.png")
ads_crosses = get_cards_tablet.find_close_ad_crosses(game_img)
ads_crosses = list(filter(not_in_ignore_region, ads_crosses))

In [ ]:
ads_crosses

In [ ]:
f, ax = plt.subplots(1, 1)

ax.imshow(game_img)
ad_crosses_xy = np.array(ads_crosses)

ax.scatter(ad_crosses_xy[:, 0], ad_crosses_xy[:, 1], c='r', marker='x')
plt.show()
